# Chapter 8 — Worked Example: Physics-Informed Inversion of the Theis Problem

**AI for Hydrogeologists** — companion notebook

**Explicitly synthetic**, unlike every other worked example in this book.
Parameters are loosely inspired by published ranges for the Albian
sandstone (Continental Intercalaire) of the northern Sahara, but this is a
controlled benchmark against a KNOWN analytical solution (Theis, 1935),
not a real field dataset — exactly as originally scoped in the book
outline for this chapter.

**Governing equation** (radial, confined, homogeneous, isotropic aquifer):

    S * ds/dt = T * (d2s/dr2 + (1/r) * ds/dr)

Analytical solution: s(r,t) = Q/(4*pi*T) * W(u), u = r^2*S/(4*T*t), where
W is the Theis well function (exponential integral E1).

In [ ]:
!pip install torch -q
import numpy as np
import torch
import torch.nn as nn
from scipy.special import exp1
import matplotlib.pyplot as plt

torch.manual_seed(42)
np.random.seed(42)


## Synthetic "true" aquifer and pumping test

In [ ]:
T_true = 450.0      # m^2/day
S_true = 8.0e-4
Q = 2000.0           # m^3/day

def theis_drawdown(r, t, T, S, Q):
    u = (r**2 * S) / (4 * T * t)
    return Q / (4 * np.pi * T) * exp1(u)

obs_radii = [15.0, 40.0, 90.0]
obs_times = np.geomspace(0.05, 5.0, 12)

obs_r, obs_t, obs_s = [], [], []
rng = np.random.default_rng(7)
for r in obs_radii:
    s_true = theis_drawdown(r, obs_times, T_true, S_true, Q)
    noise = rng.normal(0, 0.03 * s_true.mean(), size=s_true.shape)
    obs_r.extend([r]*len(obs_times)); obs_t.extend(obs_times.tolist()); obs_s.extend((s_true+noise).tolist())

obs_r = np.array(obs_r); obs_t = np.array(obs_t); obs_s = np.array(obs_s)
print(f"{len(obs_s)} synthetic observations from {len(obs_radii)} monitoring wells")
print(f"True parameters: T = {T_true} m2/day, S = {S_true:.1e}")


## PINN definition

In [ ]:
R_MIN, R_MAX, T_MAX = 5.0, 150.0, 5.0

class PINN(nn.Module):
    def __init__(self, n_hidden=32, n_layers=4):
        super().__init__()
        layers = [nn.Linear(2, n_hidden), nn.Tanh()]
        for _ in range(n_layers - 1):
            layers += [nn.Linear(n_hidden, n_hidden), nn.Tanh()]
        layers += [nn.Linear(n_hidden, 1)]
        self.net = nn.Sequential(*layers)
        # Deliberately poor initial guesses, far from the truth
        self.log_T = nn.Parameter(torch.tensor(np.log(50.0), dtype=torch.float32))
        self.log_S = nn.Parameter(torch.tensor(np.log(1e-2), dtype=torch.float32))

    def forward(self, r, t):
        x = torch.cat([r/R_MAX, t/T_MAX], dim=1)
        return self.net(x)

    @property
    def T(self): return torch.exp(self.log_T)
    @property
    def S(self): return torch.exp(self.log_S)

model = PINN()
# Separate learning rates: see the diagnostic discussion below for why
# this matters more than it might seem.
optimizer = torch.optim.Adam([
    {"params": model.net.parameters(), "lr": 2e-3},
    {"params": [model.log_T, model.log_S], "lr": 5e-5},
])

obs_r_t = torch.tensor(obs_r, dtype=torch.float32).view(-1,1)
obs_t_t = torch.tensor(obs_t, dtype=torch.float32).view(-1,1)
obs_s_t = torch.tensor(obs_s, dtype=torch.float32).view(-1,1)

def sample_collocation(n=2000):
    r = torch.rand(n,1)*(R_MAX-R_MIN)+R_MIN
    t = torch.rand(n,1)*T_MAX+1e-3
    r.requires_grad_(True); t.requires_grad_(True)
    return r, t

def pde_residual(model, r, t):
    s = model(r, t)
    s_r = torch.autograd.grad(s, r, grad_outputs=torch.ones_like(s), create_graph=True)[0]
    s_rr = torch.autograd.grad(s_r, r, grad_outputs=torch.ones_like(s_r), create_graph=True)[0]
    s_t = torch.autograd.grad(s, t, grad_outputs=torch.ones_like(s), create_graph=True)[0]
    return model.S*s_t - model.T*(s_rr + s_r/r)

r_ic = torch.rand(300,1)*(R_MAX-R_MIN)+R_MIN
t_ic = torch.full_like(r_ic, 1e-3)
t_bc = torch.rand(300,1)*T_MAX+1e-3
r_bc = torch.full_like(t_bc, R_MAX)


## Training

In [ ]:
N_EPOCHS = 8000
history = []
for epoch in range(N_EPOCHS):
    optimizer.zero_grad()
    r_col, t_col = sample_collocation()
    loss_pde = torch.mean(pde_residual(model, r_col, t_col)**2)
    loss_data = torch.mean((model(obs_r_t, obs_t_t) - obs_s_t)**2)
    loss_ic = torch.mean(model(r_ic, t_ic)**2)
    loss_bc = torch.mean(model(r_bc, t_bc)**2)
    loss = 50*loss_data + 2000*loss_pde + 10*loss_ic + 10*loss_bc
    loss.backward()
    optimizer.step()
    if epoch % 1000 == 0 or epoch == N_EPOCHS-1:
        T_est, S_est = model.T.item(), model.S.item()
        history.append((epoch, loss.item(), T_est, S_est))
        print(f"epoch {epoch:5d} | loss={loss.item():.4f} | T={T_est:7.1f} (true {T_true}) | S={S_est:.2e} (true {S_true:.1e})")


## Final evaluation

In [ ]:
T_final, S_final = model.T.item(), model.S.item()
T_err = 100*(T_final-T_true)/T_true
S_err = 100*(S_final-S_true)/S_true
print(f"Recovered T = {T_final:.1f} m2/day (true {T_true}, error {T_err:+.0f}%)")
print(f"Recovered S = {S_final:.2e} (true {S_true:.1e}, error {S_err:+.0f}%)")

r_val = np.linspace(R_MIN, R_MAX, 40)
t_val = np.geomspace(0.05, T_MAX, 40)
RR, TT = np.meshgrid(r_val, t_val)
s_analytical = theis_drawdown(RR, TT, T_true, S_true, Q)
with torch.no_grad():
    r_flat = torch.tensor(RR.ravel(), dtype=torch.float32).view(-1,1)
    t_flat = torch.tensor(TT.ravel(), dtype=torch.float32).view(-1,1)
    s_pred = model(r_flat, t_flat).numpy().reshape(RR.shape)
rmse = np.sqrt(np.mean((s_pred-s_analytical)**2))
print(f"\nDrawdown field RMSE (PINN vs analytical): {rmse:.3f} m "
      f"(mean analytical drawdown: {s_analytical.mean():.3f} m)")


## Visualization: PINN vs analytical drawdown field

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
im0 = axes[0].pcolormesh(RR, TT, s_analytical, cmap="viridis", shading="auto")
axes[0].set_title("Analytical (Theis)"); plt.colorbar(im0, ax=axes[0], label="drawdown (m)")
im1 = axes[1].pcolormesh(RR, TT, s_pred, cmap="viridis", shading="auto")
axes[1].set_title("PINN prediction"); plt.colorbar(im1, ax=axes[1], label="drawdown (m)")
im2 = axes[2].pcolormesh(RR, TT, s_pred-s_analytical, cmap="RdBu", shading="auto", vmin=-1, vmax=1)
axes[2].set_title("Difference (PINN - analytical)"); plt.colorbar(im2, ax=axes[2], label="m")
for ax in axes:
    ax.set_xlabel("r (m)"); ax.set_ylabel("t (days)")
plt.tight_layout()
plt.show()


## Honest failure diagnosis: parameter identifiability

Inverse parameter recovery did **not** succeed — T and S are both wrong by
a large margin, despite four orders of magnitude more physics-loss weight
than a first attempt and separate (slower) learning rates for the physical
parameters. A targeted diagnostic — initializing (T, S) **exactly at their
true values** and checking whether training holds them there — showed the
truth is **not a stable point** of this joint training scheme: the
optimizer drifts away even from a perfect start. This rules out poor
initialization or a simple loss-weight imbalance as the sole cause.

**What is actually happening:** early in training, before the network has
learned the correct functional shape of s(r,t), it is numerically easier
for the optimizer to reduce the loss by moving the two scalar parameters
(T, S) than by reshaping the many-weight network — dragging the parameter
estimates away from the truth before the network catches up. Meanwhile the
drawdown-field RMSE (~0.45 m against a mean drawdown of ~1.8 m) looks
superficially reasonable — a genuinely dangerous combination: **a
plausible-looking head field produced from badly wrong physical
parameters**, exactly the failure mode Section 8.1 warns about for
black-box models, now shown to also threaten physics-informed ones when
the physical parameters and the network are trained jointly without care.

This matches, with a fully worked and diagnosed example, the caution
already in Section 8.6 ("loss weighting sensitivity... among the
principal limitations" of PINNs). Documented remedies in the PINN
literature include a two-stage schedule (pre-training the network alone
with frozen parameters before releasing them), curriculum/annealed loss
weighting, and denser or better-placed observation data. None are pursued
further here, in the interest of reporting the failure honestly rather
than tuning until a preferred number appears.

---

# Part 2 — A real surrogate model: emulating the calibrated Hennaya MODFLOW model

Unlike Part 1 above (and every other example in this section), this part
uses **real, published, calibrated model data**: the actual conductivity
(K), storage (S), and geometry fields, together with the actual simulated
head fields, from the peer-reviewed calibrated MODFLOW model of the
Hennaya plain aquifer (Laoufi et al., 2024, *Sustainability*, 16, 10777;
steady-state R2 = 0.99, transient R2 = 0.987, independent validation
R2 = 0.978 against the withheld 2012 campaign).

**Task (Section 8.3):** train a machine learning surrogate that emulates
the MODFLOW model's mapping from calibrated hydraulic parameters and
position to simulated head — the "emulation" surrogate-modelling task,
using the model's own train/validation split for an apples-to-apples
comparison: 1981 (steady-state) and 2022 (transient) for training, 2012
held out entirely for validation, exactly as in the original paper.

In [ ]:
import pandas as pd
import numpy as np
import io
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

BASE = "https://raw.githubusercontent.com/Dr-LAOUFIAbdessalam/ai-hydrogeologists/main/ch08_physics_informed/data/raw/"

def load_layer_from_url(url, layer_name="Layer 1"):
    import requests
    text_all = requests.get(url).text
    lines = text_all.splitlines()
    layer_starts = {l.strip(): i for i, l in enumerate(lines) if l.startswith("Layer")}
    all_starts = sorted(layer_starts.values())
    start = layer_starts[layer_name]
    later = [s for s in all_starts if s > start]
    end = later[0] if later else len(lines)
    text = "\n".join(lines[start+1:end])
    df = pd.read_csv(io.StringIO(text), sep="\t")
    return df.loc[:, ~df.columns.str.match("Unnamed")]

geo = load_layer_from_url(BASE + "geometry.TXT")
cond = load_layer_from_url(BASE + "Conductivity.TXT")
stor = load_layer_from_url(BASE + "Storage.TXT")

static = geo.copy()
static["Kx"] = cond["Kx"]; static["Ky"] = cond["Ky"]; static["Kz"] = cond["Kz"]
static["Ss"] = stor["Ss"]; static["Sy"] = stor["Sy"]
static["PorEff"] = stor["PorEff."]; static["PorTot"] = stor["PorTot."]

def build_year_table(year, url):
    h = load_layer_from_url(url)
    df = static.copy()
    df["Head"] = h["Head"].values
    df["year"] = year
    return df[df["Head"] < 1e29].copy()  # drop MODFLOW inactive/dry cells

df1981 = build_year_table(1981, BASE + "Simulated_head_1981.TXT")
df2012 = build_year_table(2012, BASE + "Simulated_head_2012.TXT")
df2022 = build_year_table(2022, BASE + "Simulated_head_2022.TXT")
print(f"Active cells: 1981={len(df1981)}, 2012={len(df2012)}, 2022={len(df2022)}")


## Train on 1981+2022, validate on 2012 (held out entirely)

In [ ]:
for df in [df1981, df2012, df2022]:
    for c in ["Kx", "Ky", "Kz"]:
        df[f"log_{c}"] = np.log10(df[c].clip(lower=1e-15))

feature_cols = ["X","Y","Z","Top","Bot","Thick.","log_Kx","log_Ky","log_Kz",
                "Ss","Sy","PorEff","PorTot","year"]

train_df = pd.concat([df1981, df2022], ignore_index=True)
val_df = df2012.copy()
X_train, y_train = train_df[feature_cols].values, train_df["Head"].values
X_val, y_val = val_df[feature_cols].values, val_df["Head"].values

models = {"Linear Regression": LinearRegression(),
          "Random Forest": RandomForestRegressor(n_estimators=500, max_depth=12, random_state=42)}

results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_val)
    rmse = np.sqrt(mean_squared_error(y_val, pred))
    r2 = r2_score(y_val, pred)
    results[name] = (rmse, r2, pred)
    print(f"{name}: RMSE={rmse:.2f} m | R2={r2:.4f}")

print(f"\nPublished MODFLOW model's own validation on the same 2012 campaign: R2 = 0.978")


**Result: the ML surrogate (R2 ≈ 0.97) comes remarkably close to the
physically-based calibrated model's own validation performance (R2 =
0.978)** — a genuinely strong outcome, on real data, using the same
held-out validation campaign as the peer-reviewed original.

## Feature importance — and a collinearity trap

In [ ]:
rf = models["Random Forest"]
importances = pd.Series(rf.feature_importances_, index=feature_cols).sort_values(ascending=False)
print(importances.round(3))


Elevation-related features (`Top`, `Z`) dominate; the hydraulic
parameters (`log_Kx`, `Ss`, `Sy`...) show near-zero importance. Taken at
face value this would suggest the surrogate ignores the calibrated
hydraulic properties entirely - a concerning finding for a chapter about
physics-informed modelling. Before accepting that conclusion, check
whether it survives removing elevation from the model:

In [ ]:
corr_top = train_df["Head"].corr(train_df["Top"])
corr_k = train_df["Head"].corr(train_df["log_Kx"])
print(f"Correlation(Head, Top):    {corr_top:.3f}")
print(f"Correlation(Head, log_Kx): {corr_k:.3f}")

feature_cols_no_elev = ["log_Kx","log_Ky","log_Kz","Ss","Sy","PorEff","PorTot","year"]
rf_noelev = RandomForestRegressor(n_estimators=500, max_depth=12, random_state=42)
rf_noelev.fit(train_df[feature_cols_no_elev].values, y_train)
pred_noelev = rf_noelev.predict(val_df[feature_cols_no_elev].values)
print(f"\nR2 using ONLY hydraulic parameters (no position/elevation at all): "
      f"{r2_score(y_val, pred_noelev):.3f}")


**Resolution: K/S alone still explain ~93% of head variance** — they are
not useless. The near-zero importance in the full model is a classic
multicollinearity artifact: elevation and the calibrated hydraulic zones
are highly correlated (both were established from the same geological
zonation during the original calibration), so a tree-based model
preferentially splits on the smoother elevation signal once available,
making K/S appear redundant even though they carry most of the same
information. This is exactly the kind of misleading situation that
motivates moving beyond raw `feature_importances_` to SHAP-based analysis
— taken up properly in Chapter 9.